# Gerar o modelo do protótipo (Streamlit)

Roda o experimento no Colab e produz o `modelo_streamlit.joblib` que a aplicação em `app/` carrega.
Não precisa de GitHub: o repositório é enviado como `.zip`.

Execute as células na ordem (*Ambiente de execução → Executar tudo*). A célula 3 é a única que exige
decisão sua: é onde entram os arquivos da ANTAQ.

## 1. Drive e pastas

In [ ]:
import os, glob, shutil, sys, zipfile

from google.colab import drive
drive.mount("/content/drive")

SAIDA = "/content/drive/MyDrive/tcc_tempo_atracado/resultados"   # resultados e modelos treinados
DADOS_DRIVE = "/content/drive/MyDrive/tcc_tempo_atracado/dados"  # ANTAQ guardada no Drive
os.makedirs(SAIDA, exist_ok=True)
os.makedirs(DADOS_DRIVE, exist_ok=True)

cache = os.path.join(SAIDA, "modelos")
print("Modelos em cache:", sorted(os.listdir(cache)) if os.path.isdir(cache)
      else "(nenhum — a busca de hiperparâmetros vai rodar inteira, 1–2 h)")
print("ANTAQ no Drive:", sorted(os.listdir(DADOS_DRIVE)) or "(vazio)")

## 2. Repositório

Envie o `tempo-atracado-portos.zip` quando a caixa de seleção aparecer.

In [ ]:
from google.colab import files

if not os.path.isdir("/content/repo"):
    enviados = files.upload()                       # selecione o .zip do repositório
    nome = next(n for n in enviados if n.endswith(".zip"))
    with zipfile.ZipFile(nome) as zf:
        zf.extractall("/content/extraido")
    raiz = next(d for d, _, arqs in os.walk("/content/extraido") if "pyproject.toml" in arqs)
    shutil.move(raiz, "/content/repo")
    os.remove(nome)                                 # para não ser confundido com um arquivo de dados

%cd /content/repo
!pip install -q catboost xgboost shap
sys.path.insert(0, "/content/repo/src")
print(sorted(os.listdir("/content/repo")))

## 3. Arquivos da ANTAQ

São necessários seis arquivos — **Atracação** e **Carga** de 2022, 2023 e 2024 — do
[Estatístico Aquaviário](https://web3.antaq.gov.br/ea/sense/download.html). Escolha **uma** das três
formas abaixo; a célula procura nessa ordem e mostra no fim o que encontrou.

**a) Baixar dentro do Colab (mais rápido).** Abra a página do Estatístico Aquaviário, selecione o ano
e clique com o botão direito nos botões de download de *Atracação* e de *Carga* → *Copiar endereço do
link*. Cole os seis links na lista `LINKS` abaixo. A transferência acontece entre servidores, sem
passar pela sua internet.

**b) Já estão no Drive.** Se você copiou os `.zip` para `MyDrive/tcc_tempo_atracado/dados`, deixe
`LINKS` vazia: eles são usados automaticamente.

**c) Enviar do computador.** Deixe `LINKS` vazia e o Drive sem os arquivos: aparece uma caixa de
seleção. São alguns gigabytes e a conexão costuma cair no meio — prefira (a) ou (b) se puder. O que
for enviado é copiado para o Drive, então isso acontece uma única vez.

In [ ]:
LINKS = [
    # "https://.../2022Atracacao.zip",
    # "https://.../2022Carga.zip",
    # ... e assim por diante, os seis arquivos
]

ESPERADOS = [f"{ano}{tipo}" for ano in (2022, 2023, 2024) for tipo in ("Atracacao", "Carga")]
os.makedirs("dados", exist_ok=True)


def encontrados():
    """Quais dos seis arquivos já estão em dados/ (ignora maiúsculas e acentos no nome)."""
    nomes = [a.lower() for a in os.listdir("dados")]
    return {e: any(e.lower() in n for n in nomes) for e in ESPERADOS}


# (a) links colados acima
for url in LINKS:
    destino = os.path.join("dados", url.split("/")[-1].split("?")[0])
    if not os.path.exists(destino):
        !wget -q --show-progress -O "{destino}" "{url}"

# (b) cópias guardadas no Drive
for arq in glob.glob(os.path.join(DADOS_DRIVE, "*.zip")) + glob.glob(os.path.join(DADOS_DRIVE, "*.txt")):
    destino = os.path.join("dados", os.path.basename(arq))
    if not os.path.exists(destino):
        shutil.copy(arq, destino)

# (c) envio pelo navegador, só se ainda faltar alguma coisa
if not all(encontrados().values()):
    print("Faltam:", [e for e, ok in encontrados().items() if not ok])
    enviados = files.upload()
    for nome in enviados:
        shutil.move(nome, os.path.join("dados", nome))

# Guarda no Drive o que veio de fora dele
for arq in os.listdir("dados"):
    if not os.path.exists(os.path.join(DADOS_DRIVE, arq)):
        shutil.copy(os.path.join("dados", arq), os.path.join(DADOS_DRIVE, arq))

for nome, ok in encontrados().items():
    print(f"  {nome:20s} {'OK' if ok else 'FALTANDO'}")
print("\nEm dados/:", sorted(os.listdir("dados")))
assert all(encontrados().values()), "Faltam arquivos da ANTAQ — reveja as opções (a), (b) e (c) acima."

## 4. Execução

Mesma configuração do artigo. Com os modelos em cache no Drive, a busca de hiperparâmetros é pulada e
a execução cai para poucos minutos; do zero, leva de uma a duas horas.

In [ ]:
from tempo_atracado import Config, executar

cfg = Config(
    dir_dados="dados",
    dir_saida=SAIDA,
    corte_no_teste=False,
    rodar_ablacao=True,
    rodar_xgb_nativo=False,
    rodar_sementes=False,
    mostrar_figuras=True,
)
resumo = executar(cfg)
resumo

## 5. Baixar o modelo do protótipo

In [ ]:
caminho = os.path.join(SAIDA, "modelo_streamlit.joblib")
print(caminho, "—", round(os.path.getsize(caminho) / 1e6, 1), "MB")
files.download(caminho)

Coloque o arquivo baixado em `resultados/modelo_streamlit.joblib`, na pasta do repositório no seu
computador, e rode a aplicação:

```bash
pip install -e ".[app]"
streamlit run app/app.py
```

A célula abaixo baixa todo o resto (tabelas e figuras atualizadas) num zip.

In [ ]:
shutil.make_archive("/content/resultados", "zip", SAIDA)
files.download("/content/resultados.zip")